# Zarr Patch Array — Setup Notebook (Contiguous Read Benchmark)

## Schema Description

This notebook creates and seeds the Zarr array used for the
**Batch read of 1000 contiguous patches (Zarr)** benchmark.

### Array Design

| Component    | Value                                                           |
| ------------ | --------------------------------------------------------------- |
| Array type   | Zarr N-D array (4-D)                                            |
| Shape        | `(1_000_000, 32, 32, 3)` — 1M patches, each 32×32 RGB          |
| Chunks       | `(1024, 32, 32, 3)` — 1024 patches per chunk                   |
| dtype        | uint8                                                           |
| Compressor   | Zarr default (Blosc/LZ4)                                        |
| patch_id     | First axis; matches `id` PK of the relational patch table       |

### Rationale

- **Chunk size 1024 along patch_id** allows a contiguous read of 1000 patches
  to touch at most 2 chunks (since 1000 < 1024), minimising IO overhead.
- **patch_id as first axis** enables O(1) contiguous slice access (`z[start:end]`).
- **uint8** is the natural datatype for raw pixel data in histologic images.

### Setup Steps

1. Drop/recreate the Zarr array (idempotent).
2. Seed 1,000,000 patches (32×32×3, uint8) in batches of 10,000.
3. Provide a teardown cell that removes the array after benchmarking.


In [ ]:
# ── Dependencies ──────────────────────────────────────────────────────────────
import os
import shutil
import time

import numpy as np
import zarr

print(f"zarr  version : {zarr.__version__}")
print(f"NumPy version : {np.__version__}")

# ── Config ────────────────────────────────────────────────────────────────────
ZARR_PATH   = "/tmp/zarr_contiguous_patch_benchmark_1M.zarr"
NUM_PATCHES = 1_000_000
PATCH_H     = 32
PATCH_W     = 32
PATCH_C     = 3
CHUNK_SIZE  = 1024        # chunks along patch_id axis
SEED_BATCH  = 10_000      # write N patches per iteration to cap memory

print(f"\nArray path     : {ZARR_PATH}")
print(f"Total patches  : {NUM_PATCHES:,}")
print(f"Patch shape    : ({PATCH_H}, {PATCH_W}, {PATCH_C})")
estimated_gb = NUM_PATCHES * PATCH_H * PATCH_W * PATCH_C / 1e9
print(f"Uncompressed   : ~{estimated_gb:.1f} GB")

In [ ]:
# ── Idempotent array creation ─────────────────────────────────────────────────
if os.path.exists(ZARR_PATH):
    print(f"Removing existing array: {ZARR_PATH}")
    shutil.rmtree(ZARR_PATH)

os.makedirs(os.path.dirname(ZARR_PATH), exist_ok=True)

z = zarr.open(
    ZARR_PATH,
    mode='w',
    shape=(NUM_PATCHES, PATCH_H, PATCH_W, PATCH_C),
    chunks=(CHUNK_SIZE, PATCH_H, PATCH_W, PATCH_C),
    dtype='uint8',
)

print(f"Created Zarr array at: {ZARR_PATH}")
print(f"Array shape  : {z.shape}")
print(f"Chunk shape  : {z.chunks}")
print(f"dtype        : {z.dtype}")

In [ ]:
# ── Seed data ─────────────────────────────────────────────────────────────────
rng = np.random.default_rng(2024)

t_seed_start = time.perf_counter()
print("Seeding data...")

for batch_start in range(0, NUM_PATCHES, SEED_BATCH):
    batch_end = min(batch_start + SEED_BATCH, NUM_PATCHES)
    chunk = rng.integers(
        0, 256,
        size=(batch_end - batch_start, PATCH_H, PATCH_W, PATCH_C),
        dtype=np.uint8,
    )
    z[batch_start:batch_end] = chunk

    if batch_start % 100_000 == 0:
        elapsed = time.perf_counter() - t_seed_start
        print(f"  {batch_start:>9,} / {NUM_PATCHES:,}  ({elapsed:.1f}s elapsed)")

t_seed_total = time.perf_counter() - t_seed_start
print(f"Seeding complete — {t_seed_total:.1f}s total")

In [ ]:
# ── Verify array integrity ────────────────────────────────────────────────────
z_verify = zarr.open(ZARR_PATH, mode='r')
sample = z_verify[0:5]
print(f"Sample read shape  : {sample.shape}")
print(f"Sample dtype       : {sample.dtype}")
print(f"Sample value range : [{sample.min()}, {sample.max()}]")
print("Array is ready for benchmarking.")

In [ ]:
# ── Teardown (run after benchmarking) ─────────────────────────────────────────
# Uncomment to remove the array once benchmarking is complete.
# import shutil
# if os.path.exists(ZARR_PATH):
#     shutil.rmtree(ZARR_PATH)
#     print(f"Removed Zarr array: {ZARR_PATH}")